In [3]:
import requests
import time

BASE_URL = "https://store.playstation.com/valkyrie-api"
LANG = "en"
REGION = "GB"   # use GB or US
STORE = "999"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://store.playstation.com",
}

session = requests.Session()
session.headers.update(HEADERS)


def search_page(start: int, size: int = 30):
    """
    Fetch one page of the PlayStation Store catalog using search.
    Empty query returns full catalog.
    """
    url = f"{BASE_URL}/{LANG}/{REGION}/{STORE}/search"
    params = {
        "query": "",
        "size": size,
        "start": start,
    }

    r = session.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()


def crawl_all_product_ids():
    start = 0
    size = 30
    all_products = {}

    while True:
        print(f"Fetching search page start={start}")
        data = search_page(start, size)

        items = data.get("data", [])
        if not items:
            print("End of catalog reached.")
            break

        for item in items:
            product_id = item.get("id")
            name = item.get("name")
            product_type = item.get("type")

            # Deduplicate
            if product_id not in all_products:
                all_products[product_id] = {
                    "id": product_id,
                    "name": name,
                    "type": product_type,
                    "platforms": item.get("platforms"),
                }
                print(f"✓ {name}")

        start += size
        time.sleep(1)  # be polite

    return list(all_products.values())


if __name__ == "__main__":
    products = crawl_all_product_ids()
    print(f"\nTotal products collected: {len(products)}")


Fetching search page start=0


HTTPError: 404 Client Error: Not Found for url: https://store.playstation.com/valkyrie-api/en/GB/999/search?query=&size=30&start=0